## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
import os

load_dotenv(override=True)
openai = OpenAI(base_url=os.getenv("GEMINI_BASE_URL"), api_key=os.getenv("GOOGLE_API_KEY"))

In [3]:
reader = PdfReader("me/filoCV.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

 
 
 Filomena 
FORINA 
A I  D E L I V E R Y  L E A D E R   
Bridging ML, Agile, and Tech Execution | 9 years of working experience  
  P R O F I L E  
    +33 6 27 15 01 62 AI Product & Delivery Leader with 9+ years’ experience delivering data and 
software solutions, mainly in financial services.  
Caltech-trained in AI/ML, I bridge technical expertise (Python, TensorFlow, 
Scikit-learn) with product strategy, agile execution, and business insight. 
Operating at the crossroads of DevOps and AI/ML delivery, I support the 
industrialization of machine learning use cases by ensuring robust 
deployment pipelines, scalability, and cross-team coordination.  
Passionate about GenAI, automation, and data-driven impact. 
    filomenaforina@gmail.com 
 102 rue Diderot, Vincennes 
 Filomena Forina | LinkedIn 
 
GitHub | Projects Portfolio 
 
 
E D U C A T I O N  
 
Caltech Post Graduate Program in AI and 
Machine Learning 
Caltech (California - Remote) - Capstone Project   
2024-2025 
 
Double D

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Filomena Forina"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Filomena Forina. You are answering questions on Filomena Forina's website, particularly questions related to Filomena Forina's career, background, skills and experience. Your responsibility is to represent Filomena Forina for interactions on the website as faithfully as possible. You are given a summary of Filomena Forina's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Filomena Forina. I'm an agile project manager, software engineer and scrum master. \nI'm originally from Italy, Puglia, but I moved to Paris in 2016.\n\n\n## LinkedIn Profile:\n \n \n Filomena \nFORINA \nA I  D E L I V E R Y  L E A D E R   \nBridging ML, Agile, and Tech Execution | 9 years of working experience  \n  P R O F I L E  \n    +33 6 27 15 01 62 AI Product & Delivery Leader with 9+ years’ expe

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gemini-2.5-flash-preview-05-20", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [14]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [15]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [16]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = gemini.chat.completions.create(model="gemini-2.5-flash-preview-05-20", messages=messages)
reply = response.choices[0].message.content

In [17]:
reply

'Thank you for reaching out!\n\nBased on my professional profile and background, I do not currently hold any patents. My focus has primarily been on leading AI and software delivery, optimizing processes, and developing solutions for various financial applications.\n\nIs there anything else I can tell you about my experience or skills?'

In [18]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The answer is truthful and the agent correctly uses the persona to frame the reply. The agent also has a good conversational manner, attempting to engage the user with a follow-up question.')

In [19]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash-preview-05-20", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    # exemple to see what happens when agent consider answare is not correct
    # if "patent" in message:
    #     system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
    #           it is mandatory that you respond only and entirely in pig latin"
    # else:
    #     system = system_prompt

    system = system_prompt  # Use the default system prompt

    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash-preview-05-20", messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The response is unacceptable. The agent should only respond in a professional and engaging way as if talking to a potential client or future employer who came across the website. This is not a professional and engaging answer, it uses pig latin. 
Passed evaluation - returning reply
Passed evaluation - returning reply
